In [ ]:
import os
import sys
import json
import time
import kaggle
from kagglehub.competition import competition_download

import numpy as np
import pandas as pd


from sklearn.model_selection import StratifiedKFold, train_test_split
import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report
import optuna


kaggle_config_dir = os.environ.get("KAGGLE_CONFIG_DIR", os.path.expanduser("~/.config/kaggle"))
with open(os.path.join(kaggle_config_dir, "kaggle.json")) as f:
    _kaggle_creds = json.load(f)
os.environ.setdefault("KAGGLE_USERNAME", _kaggle_creds["username"])
os.environ.setdefault("KAGGLE_KEY", _kaggle_creds["key"])
path = competition_download('ing-hubs-turkiye-datathon')

In [3]:
customer_history = pd.read_csv(f"{path}/customer_history.csv")
customers = pd.read_csv(f"{path}/customers.csv")
referance_data = pd.read_csv(f"{path}/referance_data.csv")
referance_data_test = pd.read_csv(f"{path}/referance_data_test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv") 

In [ ]:
from metrics import recall_at_k, lift_at_k, convert_auc_to_gini, ing_hubs_datathon_metric

In [5]:
train_data = customers.merge(referance_data, "right", on="cust_id")
test_data = customers.merge(referance_data_test, "right", on="cust_id")

In [6]:
train_data = train_data.drop(["cust_id", "ref_date"], axis=1)
test_data = test_data.drop(["cust_id", "ref_date"], axis=1)

In [7]:
train_data

,gender,age,province,religion,work_type,work_sector,tenure,churn
0,F,64,NOH,U,Part-time,Technology,135,0
1,F,22,ZUI,C,Student,NaN,47,0
2,M,27,ZUI,U,Full-time,Finance,108,1
3,F,40,NOH,U,Unemployed,NaN,187,1
4,F,64,GEL,U,Part-time,Public Sector,218,0
...,...,...,...,...,...,...,...,...
133282,F,54,GEL,C,Part-time,Public Sector,217,0
133283,M,47,GEL,C,Full-time,Public Sector,37,0
133284,F,66,NOB,C,Retired,NaN,227,0
133285,F,31,ZUI,U,Self-employed,Education,156,1


In [8]:
num_cols = [col for col in test_data.columns if test_data[col].dtype != object]
cat_cols = [col for col in test_data.columns if test_data[col].dtype == object]

In [9]:
train_cat_df = pd.get_dummies(train_data[cat_cols], drop_first=True)
test_cat_df = pd.get_dummies(test_data[cat_cols], drop_first=True)

In [10]:
train_num_df = pd.DataFrame(train_data[num_cols].values, columns=num_cols)
test_num_df = pd.DataFrame(test_data[num_cols].values, columns=num_cols)

In [11]:
new_train = pd.concat([train_cat_df, train_num_df,train_data["churn"]], axis=1)
new_test  =pd.concat([test_cat_df, test_num_df], axis=1)

In [12]:
X = new_train.drop("churn", axis=1)
y = new_train["churn"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
def objective(trial, X, y):
    """
    Optuna'nın her bir denemede çalıştıracağı ve özel metriği maksimize edeceği fonksiyon.
    """

    # GPU'su olmayan makinelerde de çalışsın diye varsayılan 'cpu'; GPU varsa
    # XGB_DEVICE=cuda ortam değişkeniyle değiştirilebilir.
    xgb_device = os.environ.get("XGB_DEVICE", "cpu")

    # Hiperparametre Arama Uzayını Tanımla
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'booster': 'gbtree',
        'device': xgb_device,
        'early_stopping_rounds': 50,
        'n_estimators': 1000,
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }

    # Dengesiz veri için kritik olan sınıf ağırlığını hesapla
    scale_pos_weight = np.sum(y == 0) / np.sum(y == 1)
    param['scale_pos_weight'] = scale_pos_weight

    # StratifiedKFold ile Çapraz Doğrulama
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in cv.split(X, y):
        X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
        X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**param, random_state=42)
        
        # Modeli eğit (Early stopping ile aşırı öğrenmeyi engelle)
        model.fit(X_train_fold, y_train_fold,
                  eval_set=[(X_val_fold, y_val_fold)],
                  verbose=False)
        
        preds_proba = model.predict_proba(X_val_fold)[:, 1]
        
        # Özel değerlendirme metriğini kullanarak skoru hesapla
        custom_score = ing_hubs_datathon_metric(y_val_fold, preds_proba)
        scores.append(custom_score)

    # Ortalamayı döndür. Optuna bu değeri maksimize etmeye çalışacak.
    return np.mean(scores)


In [ ]:

# =============================================================================
# 5. Optimizasyon Sürecini Başlatma
# =============================================================================
print("--- Optuna Optimizasyonu Başlatılıyor ---")
# 'direction="maximize"' ile özel metriğimizin en yüksek değerini arıyoruz.
# storage + load_if_exists: kernel yeniden başlasa bile trial geçmişi kaybolmaz.
study = optuna.create_study(
    direction='maximize',
    study_name='xgb-baseline-churn-ing-metric',
    storage='sqlite:///optuna_studies_baseline.db',
    load_if_exists=True,
)

# Optimizasyonu n_trials kadar deneme ile çalıştır
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=50, show_progress_bar=True)

print("Optimizasyon tamamlandı.\n")
print("--- En İyi Optimizasyon Sonuçları ---")
best_trial = study.best_trial
print(f"En İyi Değer (Ortalama Özel Metrik): {best_trial.value:.4f}")
print("En İyi Parametreler:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")
print("-" * 30, "\n")


In [16]:

# =============================================================================
# 6. Final Modelin Eğitilmesi ve Değerlendirilmesi
# =============================================================================
print("--- Final Model Eğitiliyor ve Değerlendiriliyor ---")
# Optuna'nın bulduğu en iyi parametreleri al
best_params = best_trial.params

# Optimizasyon dışında kalan sabit parametreleri ekle
best_params['scale_pos_weight'] = np.sum(y_train == 0) / np.sum(y_train == 1)
best_params['n_estimators'] = 2000  # Early stopping için yüksek bir değer
best_params['random_state'] = 42
best_params['objective'] = 'binary:logistic'
best_params["early_stopping_rounds"] = 50

# Final modeli en iyi parametrelerle oluştur
final_model = xgb.XGBClassifier(**best_params)

# Final modelin eğitiminde de early stopping kullanmak iyi bir pratiktir.
# Bunun için eğitim verisinden küçük bir validasyon seti ayırabiliriz.
X_train_part, X_val_part, y_train_part, y_val_part = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

final_model.fit(X_train_part, y_train_part,
                eval_set=[(X_val_part, y_val_part)],
                verbose=False)

# Daha önce hiç görülmemiş TEST VERİSİ üzerinde tahmin yap
y_pred_proba_test = final_model.predict_proba(X_test)[:, 1]

# Test seti üzerinde özel metrik skorunu hesapla
final_custom_score = ing_hubs_datathon_metric(y_test, y_pred_proba_test)
print(f"Test Seti Üzerindeki Özel Metrik Skoru: {final_custom_score:.4f}\n")

# Özel metriği oluşturan alt metriklerin dökümünü de alalım
test_auc = roc_auc_score(y_test, y_pred_proba_test)
test_gini = convert_auc_to_gini(test_auc)
test_recall10 = recall_at_k(y_test, y_pred_proba_test, k=0.1)
test_lift10 = lift_at_k(y_test, y_pred_proba_test, k=0.1)

print("--- Test Seti Detaylı Metrikler ---")
print(f"Gini: {test_gini:.4f}")
print(f"Recall@10%: {test_recall10:.4f}")
print(f"Lift@10%: {test_lift10:.4f}\n")

# Sınıflandırma raporu için bir eşik değeri belirleyelim (örn: 0.5)
y_pred_class_test = (y_pred_proba_test > 0.5).astype(int)
print("--- Test Seti Classification Report (0.5 Eşik Değeri ile) ---")
print(classification_report(y_test, y_pred_class_test))
print("=" * 70)

--- Final Model Eğitiliyor ve Değerlendiriliyor ---
Test Seti Üzerindeki Özel Metrik Skoru: 0.4156

--- Test Seti Detaylı Metrikler ---
Gini: 0.0442
Recall@10%: 0.1138
Lift@10%: 1.1383

--- Test Seti Classification Report (0.5 Eşik Değeri ile) ---
              precision    recall  f1-score   support

           0       0.86      0.55      0.67     28604
           1       0.15      0.48      0.23      4718

    accuracy                           0.54     33322
   macro avg       0.51      0.51      0.45     33322
weighted avg       0.76      0.54      0.61     33322



In [17]:
best_params.pop("early_stopping_rounds")

50

In [18]:
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X,y)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8082353691901047
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
import joblib

joblib.dump(final_model, "xgb_final_model.joblib")
print("Final XGBoost modeli 'xgb_final_model.joblib' dosyasına kaydedildi.")

In [19]:
sample_submission["churn"] = final_model.predict_proba(new_test)[:, 1]

In [20]:
sample_submission.to_csv('/tmp/submission.csv', index=False)
kaggle.api.competition_submit(
    file_name='/tmp/submission.csv', 
    message='xgb with Optuna kfold', 
    competition='ing-hubs-turkiye-datathon'
)

100%|██████████| 713k/713k [00:01<00:00, 564kB/s] 


{"message": "Successfully submitted to ING Hubs T\u00fcrkiye Datathon", "ref": 54759441}